# Dog Breed Classification with PyTorch & MobileNetV3

This notebook provides an interactive walkthrough of the Dog Breed Classification training pipeline:
1. **Exploratory Data Analysis (EDA)**: Inspecting merged breed class distributions.
2. **Data Augmentations**: Displaying sample pre-processed training images.
3. **Model Fine-Tuning**: Initializing MobileNetV3-Large with pretrained weights.
4. **Training Loop & Evaluation**: Running validation audits and saving metrics.
5. **ONNX Export**: Saving the graph structure to compile for thin production containers.

In [ ]:
import os
import json
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torchvision import transforms

### 1. Dataset Structure Inspection
Let's confirm the dataset folders are generated and read the number of categories.

In [ ]:
dataset_dir = "./dataset"
if os.path.exists(dataset_dir):
    train_dir = os.path.join(dataset_dir, "train")
    breeds = sorted(os.listdir(train_dir))
    print(f"Total classes: {len(breeds)}")
    print(f"Sample categories: {breeds[:5]}")
else:
    print("Dataset folder not found. Run 'python download_and_merge.py --dry-run' first.")

### 2. Visualize Image Transform Augmentations
We apply random rotations and resized crops to make the model robust against camera variations.

In [ ]:
loader_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.1, 0.1, 0.1)
])

# Create a dummy blank image to visualize transforms if no folder is present
sample_image = Image.new('RGB', (300, 300), color=(128, 128, 255))
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    axes[i].imshow(loader_transform(sample_image))
    axes[i].axis('off')
plt.show()

### 3. Initialize PyTorch Model Backbone
We load MobileNetV3-Large with standard ImageNet weights.

In [ ]:
from ml.model import DogBreedClassifierModel

model = DogBreedClassifierModel(num_classes=120)
print("Model initialized. Backbone classifier signature:")
print(model.backbone.classifier)

### 4. Execute Dry-Run Offline Training & ONNX Compiler
To verify full execution health, let's run the training runner script with mock dataset inputs.

In [ ]:
!python download_and_merge.py --dry-run
!python train.py --data-dir ./ml/dataset --epochs 1